# DoorDash Delivery ETA — Etapa 07: Avaliação Final no Conjunto de Teste

Este notebook executa a **avaliação final única** do modelo congelado na Etapa 06.2 (`HistGradientBoostingRegressor` com `loss="absolute_error"`, `max_iter=20`, `early_stopping=False`) utilizando o conjunto de teste (`30.041` registros), mantendo todos os registros (sem remoção de outliers) e sem realizar novos ajustes ou seleções.

In [ ]:
import json
import platform
import sys
import time
from datetime import datetime
from pathlib import Path

import joblib
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import seaborn as sns
import sklearn
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.pipeline import Pipeline

sns.set_theme(style="whitegrid")
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_preparation import prepare_datasets
from src.feature_engineering import build_features
from src.temporal_validation import temporal_split_with_embargo
from src.model_training import (
    audit_temporal_split,
    build_preprocessor,
    compute_regression_metrics,
    evaluate_by_duration_band,
    prepare_feature_frame,
    CONFIRMED_MODEL_FEATURES,
    TARGET_COL,
)

RAW_PATH = PROJECT_ROOT / "data" / "raw" / "historical_data.csv"
_, _, df_model, _ = prepare_datasets(RAW_PATH)
df_feat = build_features(df_model)
split = temporal_split_with_embargo(df_feat)

audit = audit_temporal_split(split.train, split.val, split.test, split.val_start, split.test_start)
print("Auditoria temporal aprovada:", audit)
print(f"Executado em: {datetime.now().isoformat(timespec='seconds')}")
print(f"Versões: Python {platform.python_version()} | pandas {pd.__version__} | numpy {np.__version__} | scikit-learn {sklearn.__version__}")

## 1. Treinamento do Modelo Congelado e Baseline (Exclusivamente no Treino)

In [ ]:
X_train = prepare_feature_frame(split.train, CONFIRMED_MODEL_FEATURES)
y_train = split.train[TARGET_COL].values
X_val = prepare_feature_frame(split.val, CONFIRMED_MODEL_FEATURES)
y_val = split.val[TARGET_COL].values
X_test = prepare_feature_frame(split.test, CONFIRMED_MODEL_FEATURES)
y_test = split.test[TARGET_COL].values

frozen_params = {
    "loss": "absolute_error",
    "learning_rate": 0.08,
    "max_iter": 20,
    "max_leaf_nodes": 63,
    "min_samples_leaf": 30,
    "l2_regularization": 0.0,
    "early_stopping": False,
    "random_state": 42,
}

baseline_pipe = Pipeline([
    ("prep", build_preprocessor(CONFIRMED_MODEL_FEATURES, scale_numeric=False)),
    ("model", DummyRegressor(strategy="median")),
])

frozen_pipe = Pipeline([
    ("prep", build_preprocessor(CONFIRMED_MODEL_FEATURES, scale_numeric=False)),
    ("model", HistGradientBoostingRegressor(**frozen_params)),
])

baseline_pipe.fit(X_train, y_train)
t0 = time.perf_counter()
frozen_pipe.fit(X_train, y_train)
train_time = time.perf_counter() - t0
print(f"Modelo congelado treinado em {train_time:.2f}s (n_train={len(X_train)}, n_features={len(CONFIRMED_MODEL_FEATURES)})")

## 2. Avaliação no Conjunto de Teste e Comparação com Validação e Baseline

In [ ]:
y_val_base = baseline_pipe.predict(X_val)
y_val_pred = frozen_pipe.predict(X_val)
y_test_base = baseline_pipe.predict(X_test)
t1 = time.perf_counter()
y_test_pred = frozen_pipe.predict(X_test)
infer_time = time.perf_counter() - t1

m_val_base = compute_regression_metrics(y_val, y_val_base)
m_val_frozen = compute_regression_metrics(y_val, y_val_pred)
m_test_base = compute_regression_metrics(y_test, y_test_base)
m_test_frozen = compute_regression_metrics(y_test, y_test_pred)

summary_tbl = pd.DataFrame([
    {"Conjunto": "Validação", "Modelo": "Baseline (Median)", **m_val_base},
    {"Conjunto": "Validação", "Modelo": "HGB Congelado", **m_val_frozen},
    {"Conjunto": "Teste", "Modelo": "Baseline (Median)", **m_test_base},
    {"Conjunto": "Teste", "Modelo": "HGB Congelado", **m_test_frozen},
])
display(summary_tbl)
print(f"Tempo de inferência no teste (n={len(X_test)}): {infer_time:.3f}s")

## 3. Diagnóstico de Erros no Conjunto de Teste (`n = 30.041`)

In [ ]:
abs_err = np.abs(y_test - y_test_pred)
print(f"Previsões negativas (< 0): {(y_test_pred < 0).sum()}")
for lim in [30, 60, 90, 120]:
    cnt = int((abs_err > lim).sum())
    print(f"Erros absolutos > {lim} min: {cnt} ({cnt/len(y_test)*100:.2f}%)")

band_test = evaluate_by_duration_band(split.test[TARGET_COL], y_test_pred)
display(band_test)

test_diag = split.test[["created_at", "actual_delivery_time", "market_id", "store_id", "store_primary_category", "total_items", "subtotal"]].copy()
test_diag["y_true_min"] = y_test
test_diag["y_pred_min"] = np.round(y_test_pred, 2)
test_diag["abs_err_min"] = np.round(abs_err, 2)
display(test_diag.sort_values("abs_err_min", ascending=False).head(20))

## 4. Visualizações Finais

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 11))

m180 = y_test <= 180
axes[0, 0].scatter(y_test[m180], y_test_pred[m180], alpha=0.12, s=8, color="#2b5c8f")
axes[0, 0].plot([0, 180], [0, 180], "r--", linewidth=1.5)
axes[0, 0].set_title("Teste: Reais vs Previstos (<= 180 min)")
axes[0, 0].set_xlabel("Duração Real (min)")
axes[0, 0].set_ylabel("Duração Prevista (min)")

res_test = y_test - y_test_pred
sns.histplot(res_test[(res_test >= -60) & (res_test <= 120)], bins=70, kde=True, ax=axes[0, 1], color="#4c72b0")
axes[0, 1].axvline(0, color="red", linestyle="--")
axes[0, 1].set_title("Teste: Distribuição dos Resíduos (y_real - y_prev)")
axes[0, 1].set_xlabel("Resíduo (min)")

band_test["MAE"].plot(kind="bar", ax=axes[1, 0], color="#dd8452", label="MAE")
band_test["MedAE"].plot(kind="line", marker="o", ax=axes[1, 0], color="#2b5c8f", linewidth=2, label="MedAE")
axes[1, 0].set_title("Teste: Erro por Faixa de Duração Real")
axes[1, 0].set_ylabel("Erro (min)")
axes[1, 0].tick_params(axis="x", rotation=20)
axes[1, 0].legend()

cmp_df = pd.DataFrame({
    "Baseline (Teste)": [m_test_base["MAE"], m_test_base["RMSE"], m_test_base["MedAE"], m_test_base["P90_AE"]],
    "HGB (Val)": [m_val_frozen["MAE"], m_val_frozen["RMSE"], m_val_frozen["MedAE"], m_val_frozen["P90_AE"]],
    "HGB (Teste)": [m_test_frozen["MAE"], m_test_frozen["RMSE"], m_test_frozen["MedAE"], m_test_frozen["P90_AE"]],
}, index=["MAE", "RMSE", "MedAE", "P90_AE"])
cmp_df.plot(kind="bar", ax=axes[1, 1])
axes[1, 1].set_title("Baseline vs HGB Congelado (Val e Teste)")
axes[1, 1].set_ylabel("Erro (min)")
axes[1, 1].tick_params(axis="x", rotation=0)

fig.tight_layout()
plt.show()